# 回放分析模板

加载模拟回放 JSON，交互式分析状态变化和事件流。

In [ ]:
import json
import sys
from pathlib import Path

# 设置项目路径
PROJECT_ROOT = Path('.').resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
sys.path.insert(0, str(PROJECT_ROOT))

from harness.sim.replay import load_replay, text_replay

In [ ]:
# === 修改这里：指定回放文件路径 ===
REPLAY_FILE = PROJECT_ROOT / 'tests/modules/YOUR_MODULE/replays/YOUR_REPLAY.json'

data = load_replay(REPLAY_FILE)
print(f"总 ticks: {data['total_ticks']}")
print(f"结果: {'通过' if data.get('passed') else '未通过'}")
print(f"不变量违反: {len(data.get('invariant_violations', []))}")

In [ ]:
# 文本回放
print(text_replay(data))

In [ ]:
import matplotlib.pyplot as plt

def plot_numeric_fields(data, fields=None):
    """绘制状态中数值字段随 tick 变化的趋势图。"""
    ticks = data['ticks']
    if not ticks:
        print('无数据')
        return

    # 自动检测数值字段
    def find_numeric(obj, prefix=''):
        result = []
        if not isinstance(obj, dict):
            return result
        for k, v in obj.items():
            p = f'{prefix}.{k}' if prefix else k
            if isinstance(v, (int, float)):
                result.append(p)
            elif isinstance(v, dict):
                result.extend(find_numeric(v, p))
        return result

    if fields is None:
        fields = find_numeric(ticks[0]['state_after'])[:8]

    def get_val(obj, path):
        for k in path.split('.'):
            if isinstance(obj, dict):
                obj = obj.get(k)
            else:
                return None
        return obj

    fig, ax = plt.subplots(figsize=(12, 5))
    x = [t['tick'] for t in ticks]
    for field in fields:
        y = [get_val(t['state_after'], field) for t in ticks]
        ax.plot(x, y, label=field.split('.')[-1], marker='.')

    ax.set_xlabel('Tick')
    ax.set_ylabel('值')
    ax.set_title('状态数值变化趋势')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_numeric_fields(data)

In [ ]:
# 逐 tick 检查（修改 tick_idx 查看不同时刻）
tick_idx = 0

tick = data['ticks'][tick_idx]
print(f"=== Tick {tick['tick']} ===")
print(f"事件:")
for ev in tick.get('events', []):
    print(f"  > {ev}")
print(f"\n状态变化:")
before, after = tick['state_before'], tick['state_after']
for k in set(list(before.keys()) + list(after.keys())):
    if before.get(k) != after.get(k):
        print(f"  {k}: {before.get(k)} -> {after.get(k)}")